<a href="https://colab.research.google.com/github/soheldatta17/mid_eval_meta_model/blob/main/Meta_Model_Initial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""MetaKneeEnsemble.py  —  v3.0  (Cascaded Intermediate + Late Fusion | XGBoost Arbiter)
==========================================================================================
Meta-Model Pipeline for Knee Osteoarthritis Severity Detection.

Architecture (aligns with paper: "A Meta-Model Architecture with Cryptographic
Validation and XGBoost Optimization for Knee Osteoarthritis Diagnostics"):

  FUSION VECTOR  →  9-dimensional  (was 14D, now corrected to match paper intent)
  ┌─────────────────────────────────────────────────────────┐
  │  Severity probs  : M1(5)  [KL Grade 0-4 softmax]       │
  │  JSN bins        : M2(2)  [narrow / normal — discrete]  │
  │  Morphology      : M3(2)  [Osteophytes, Sclerosis]      │
  │  Total = 9D   Vfusion ∈ ℝ⁹                             │
  └─────────────────────────────────────────────────────────┘

  PIPELINE STAGES:
  ─────────────────────────────────────────────────────────
  Stage 0 : Cryptographic SHA-256 de-duplication
  Stage 1 : Feature extraction → Vfusion (9D)
  Stage 2 : Intermediate Fusion — Lightweight MLP
              9D → 64 → 32 → Pmeta(5D)   [CE + dropout]
  Stage 3 : Late Fusion Decision Vector
              Xboost = [Vfusion(9) ⊕ Pmeta(5)] ∈ ℝ¹⁴
  Stage 4 : XGBoost Arbiter (sole final classifier)
              Xboost(14D) → KL Grade {0,1,2,3,4}
  Stage 5 : Grad-CAM explainability on Model 1 backbone

Sub-models (loaded from Google Drive):
  - Model 1 (TorchScript .pt) : best_knee_ensemble_cbam.pt
      → 5-class severity  (EfficientNet-B5 + ResNet18 + CBAM)
  - Model 2 (Keras .keras)    : final_knee_cnn_model.keras
      → JSN regression output (2-bin discretization applied here)
  - Model 3 (TorchScript .pt) : final_mmorphattention.pt
      → 4-class morphology → we take first 2 dims (Osteophytes, Sclerosis)

Changes from v2.1:
  ✓ Feature vector compressed: 14D → 9D  (5 sev + 2 JSN + 2 morph)
  ✓ Removed RF, SVM, GBT, StackedEnsemble — XGBoost is the sole arbiter
  ✓ Proper intermediate fusion MLP → meta-probability Pmeta(5D)
  ✓ Late fusion: Xboost = Vfusion ⊕ Pmeta  (9+5=14D into XGBoost)
  ✓ Grad-CAM on Model 1 EfficientNet backbone for XAI output
  ✓ Leaner training loop, better early stopping
"""

# ==============================================================================
# STEP 0 — Install gdown if needed
# ==============================================================================
import subprocess
import sys

try:
    import gdown
except ImportError:
    print("[SETUP] Installing gdown ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "gdown", "-q"], check=True)
    import gdown

# ==============================================================================
# STEP 1 — Download Sub-Models + Kaggle Dataset Setup
# ==============================================================================
print("=" * 70)
print("[STEP 1/10] Downloading sub-models from Google Drive & Kaggle setup ...")
print("=" * 70)

import os
import zipfile
import shutil
import hashlib

# ── Google Drive model assets ─────────────────────────────────────────────────
DRIVE_ASSETS = {
    "model1": {
        "id"   : "1orbyJ0UU44HT3G8inoGstlJ0DhJlQXjj",
        "local": "/content/best_knee_ensemble_cbam.pt",
        "desc" : "Model 1 — CBAM Ensemble (TorchScript) [severity 5-class]",
    },
    "model2": {
        "id"   : "1Hr4gHki9nl6nmXPO0xsAU7FnlfqldHZ8",
        "local": "/content/final_knee_cnn_model.keras",
        "desc" : "Model 2 — Keras CNN + SE Block [JSN regression → 2-bin]",
    },
    "model3": {
        "id"   : "16ozIZmH36J0K90bY9Jfe4YDTvS2SDDPh",
        "local": "/content/final_mmorphattention.pt",
        "desc" : "Model 3 — MorphAttention (TorchScript) [morph 4-class → 2]",
    },
}

MODEL1_PATH = DRIVE_ASSETS["model1"]["local"]
MODEL2_PATH = DRIVE_ASSETS["model2"]["local"]
MODEL3_PATH = DRIVE_ASSETS["model3"]["local"]


def _gdrive_download(key: str) -> str:
    """Download Drive asset; skip if cached."""
    asset = DRIVE_ASSETS[key]
    local = asset["local"]
    if os.path.exists(local):
        mb = os.path.getsize(local) / 1e6
        print(f"  [cached] {asset['desc']}  ({mb:.1f} MB)")
        return local
    print(f"  [download] {asset['desc']} ...")
    gdown.download(f"https://drive.google.com/uc?id={asset['id']}", local, quiet=False)
    if not os.path.exists(local):
        raise RuntimeError(f"FATAL: Download failed for {asset['desc']}. "
                           f"Check Drive share settings. ID={asset['id']}")
    print(f"  ✓ saved → {local}  ({os.path.getsize(local)/1e6:.1f} MB)")
    return local


for k in ["model1", "model2", "model3"]:
    _gdrive_download(k)

# ── Kaggle credentials ────────────────────────────────────────────────────────
print("\n[STEP 1/10] Configuring Kaggle ...")
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
subprocess.run(["cp", "/content/kaggle.json", os.path.expanduser("~/.kaggle/")], check=True)
subprocess.run(["chmod", "600", os.path.expanduser("~/.kaggle/kaggle.json")], check=True)

DOWNLOAD_DIR = "/content"
POOL_DIR     = "./knee_pool"
os.makedirs(POOL_DIR, exist_ok=True)

KAGGLE_DATASETS = [
    {
        "slug"    : "hafiznouman786/annotated-dataset-for-knee-arthritis-detection",
        "zip_stem": "annotated-dataset-for-knee-arthritis-detection",
        "roots"   : ["Training", "training", "train", "."],
    },
    {
        "slug"    : "shashwatwork/knee-osteoarthritis-dataset-with-severity",
        "zip_stem": "knee-osteoarthritis-dataset-with-severity",
        "roots"   : ["train", "Train", "training", "."],
    },
    {
        "slug"    : "tommyngx/kneeoa",
        "zip_stem": "kneeoa",
        "roots"   : ["train", "Train", "."],
    },
]

seen_hashes   = set()
pool_manifest = []          # [(abs_path, grade_int)]


def _sha256_prefix(path: str, nbytes: int = 8192) -> str:
    """First-8 K SHA-256 hex for fast de-dup."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read(nbytes))
    return h.hexdigest()


def _grade_from_folder(name: str):
    import re
    m = re.match(r'^(\d+)', name.strip())
    if m and int(m.group(1)) <= 4:
        return int(m.group(1))
    grade_map = {"normal": 0, "doubtful": 1, "mild": 2, "moderate": 3, "severe": 4}
    for k, v in grade_map.items():
        if k in name.lower():
            return v
    return None


def _ingest_root(root_dir: str, tag: str) -> int:
    added = 0
    for folder in sorted(os.listdir(root_dir)):
        class_dir = os.path.join(root_dir, folder)
        if not os.path.isdir(class_dir):
            continue
        grade = _grade_from_folder(folder)
        if grade is None:
            continue
        grade_pool = os.path.join(POOL_DIR, str(grade))
        os.makedirs(grade_pool, exist_ok=True)
        for fname in os.listdir(class_dir):
            if not fname.lower().endswith((".png", ".jpg", ".jpeg")):
                continue
            src = os.path.join(class_dir, fname)
            h   = _sha256_prefix(src)
            if h in seen_hashes:
                continue
            seen_hashes.add(h)
            dst = os.path.join(grade_pool, f"{tag}_{fname}")
            shutil.copy2(src, dst)
            pool_manifest.append((dst, grade))
            added += 1
    return added


for ds in KAGGLE_DATASETS:
    slug, zip_stem, roots = ds["slug"], ds["zip_stem"], ds["roots"]
    zip_path = os.path.join(DOWNLOAD_DIR, f"{zip_stem}.zip")
    tag      = slug.split("/")[-1][:12]
    print(f"\n[STEP 1/10] Downloading: {slug} ...")
    try:
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", slug, "-p", DOWNLOAD_DIR],
            check=True
        )
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(DOWNLOAD_DIR)
    except Exception as ex:
        print(f"  WARNING: {ex}")
        continue
    for root_cand in roots:
        rp = os.path.join(DOWNLOAD_DIR, root_cand)
        if os.path.isdir(rp):
            n = _ingest_root(rp, tag)
            print(f"  Root '{root_cand}' → {n} images ingested.")
            if n > 0:
                break

from collections import Counter as _Counter
print(f"\n[STEP 1/10] Pool: {len(pool_manifest)} unique images")
print(f"  Grade dist: {dict(sorted(_Counter(g for _, g in pool_manifest).items()))}")
if not pool_manifest:
    raise RuntimeError("FATAL: No images. Check Kaggle credentials/slugs.")

# ==============================================================================
# STEP 2 — Import Libraries
# ==============================================================================
print("\n" + "=" * 70)
print("[STEP 2/10] Importing libraries ...")
print("=" * 70)

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
import random
import copy
import pickle
import warnings
warnings.filterwarnings("ignore")
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import cv2                          # for Grad-CAM resize / overlay
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, accuracy_score)
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBClassifier
except ImportError:
    subprocess.run(["pip", "install", "xgboost", "-q"], check=True)
    from xgboost import XGBClassifier

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[STEP 2/10] Device: {DEVICE}")

# ==============================================================================
# STEP 3 — Load Sub-Models
# ==============================================================================
print("\n" + "=" * 70)
print("[STEP 3/10] Loading sub-models ...")
print("=" * 70)

# Model 1 — EfficientNet-B5 + ResNet18 + CBAM  (severity 5-class)
model1 = torch.jit.load(MODEL1_PATH, map_location=DEVICE)
model1.eval()
print("[STEP 3/10] Model 1 loaded ✓  (severity | TorchScript)")

# Model 2 — Keras CNN + SE Block  (JSN regression → discretised to 2 bins)
model2_tf = tf.keras.models.load_model(MODEL2_PATH)
model2_tf.trainable = False
_m2_last_act = None
try:
    _m2_last_act = model2_tf.layers[-1].activation.__name__
except Exception:
    pass
MODEL2_HAS_SOFTMAX = (_m2_last_act == "softmax")
print(f"[STEP 3/10] Model 2 loaded ✓  (JSN | Keras | last act='{_m2_last_act}')")

# Model 3 — ResNet50 + VGG19 + CBAM  (morphology 4-class → use dim 0,1 = Osteophytes, Sclerosis)
model3 = torch.jit.load(MODEL3_PATH, map_location=DEVICE)
model3.eval()
print("[STEP 3/10] Model 3 loaded ✓  (morphology | TorchScript)")
print("[STEP 3/10] All sub-models ready.\n")

# ==============================================================================
# STEP 4 — Configuration
# ==============================================================================
print("=" * 70)
print("[STEP 4/10] Configuration ...")
print("=" * 70)

SEVERITY_NAMES     = ["Normal", "Doubtful", "Mild", "Moderate", "Severe"]
NUM_SEVERITY       = 5          # M1 output dims

# JSN: Model 2 output is treated as a raw scalar (or 5-class prob).
# We convert it to a 2-bin vector: [P(narrow), P(normal)] via sigmoid threshold.
NUM_JSN            = 2          # 2-bin discretised JSN representation

# Morphology: Model 3 emits 4 sigmoid outputs; we use dims [0,1] = Osteophytes, Sclerosis
NUM_MORPH          = 2

# ── Fusion vector Vfusion ∈ ℝ⁹ ────────────────────────────────────────────────
#   [sev(5) | jsn(2) | morph(2)]  — matches paper's 8D intent with JSN as 2-bin
FUSION_DIM         = NUM_SEVERITY + NUM_JSN + NUM_MORPH    # 9

# ── MLP intermediate fusion ────────────────────────────────────────────────────
MLP_HIDDEN         = 64        # lean: 9→64→32→5
MLP_OUT            = NUM_SEVERITY                           # Pmeta ∈ ℝ⁵

# ── XGBoost input = late fusion ───────────────────────────────────────────────
#   Xboost = [Vfusion(9) ⊕ Pmeta(5)] ∈ ℝ¹⁴
XGBOOST_INPUT_DIM  = FUSION_DIM + MLP_OUT                  # 14

BATCH_SIZE         = 32
NUM_EPOCHS         = 50
LR                 = 1e-3
WEIGHT_DECAY       = 1e-4
PATIENCE           = 10
TALLY_N            = 500

SAVE_DIR = "./meta_model_outputs_v3"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"  Fusion vector (Vfusion)  : {FUSION_DIM}D  [sev(5) | jsn(2) | morph(2)]")
print(f"  MLP intermediate output  : {MLP_OUT}D  (Pmeta)")
print(f"  XGBoost input (Xboost)   : {XGBOOST_INPUT_DIM}D  [Vfusion ⊕ Pmeta]")
print(f"  MLP hidden               : {MLP_HIDDEN}")
print(f"  Batch/Epochs/Patience    : {BATCH_SIZE}/{NUM_EPOCHS}/{PATIENCE}")
print(f"  Outputs                  : {SAVE_DIR}")

# ==============================================================================
# STEP 5 — Preprocessing Helpers
# ==============================================================================
print("\n" + "=" * 70)
print("[STEP 5/10] Preprocessing helpers ...")
print("=" * 70)

# ImageNet normalisation for PyTorch backbones
transform_pt = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def preprocess_for_pt(image_path: str) -> torch.Tensor:
    """RGB 224×224 normalised tensor → (1,3,224,224) on DEVICE."""
    img = Image.open(image_path).convert("RGB")
    return transform_pt(img).unsqueeze(0).to(DEVICE)


def preprocess_for_tf(image_path: str):
    """Grayscale 256×256 float32 → (1,256,256,1)."""
    img = tf.io.read_file(image_path)
    img = tf.image.decode_image(img, channels=1, expand_animations=False)
    img = tf.image.resize(img, [256, 256])
    img = tf.cast(img, tf.float32)
    return tf.expand_dims(img, axis=0)


# ── JSN discretisation helper ──────────────────────────────────────────────────
def jsn_to_2bin(raw_output) -> np.ndarray:
    """
    Convert Model 2 raw output to a 2-bin JSN vector.

    Model 2 was trained as a severity classifier (5-class softmax) but its
    continuous probability mass over grades 3-4 (Moderate-Severe) correlates
    strongly with joint-space narrowing.  We define:
        P(narrow) = P(grade≥2) = sum of classes 2,3,4
        P(normal) = P(grade<2)  = sum of classes 0,1
    This gives a clinically meaningful 2-bin representation consistent with
    the paper's vjsn ∈ ℝ¹ extended to a 2D probability simplex.
    """
    if MODEL2_HAS_SOFTMAX:
        probs5 = np.array(raw_output[0], dtype=np.float32)
    else:
        probs5 = tf.nn.softmax(raw_output).numpy()[0].astype(np.float32)
    p_narrow = float(probs5[2] + probs5[3] + probs5[4])  # grade ≥ 2
    p_normal = float(probs5[0] + probs5[1])               # grade < 2
    return np.array([p_narrow, p_normal], dtype=np.float32)


print("[STEP 5/10] Preprocessing helpers defined.")
print("  JSN 2-bin: P(narrow)=Σprob[grade≥2], P(normal)=Σprob[grade<2]")

# ==============================================================================
# STEP 6 — Feature Extraction  →  Vfusion ∈ ℝ⁹
# ==============================================================================
print("\n" + "=" * 70)
print("[STEP 6/10] Feature extraction — Vfusion ∈ ℝ⁹  [sev(5)|jsn(2)|morph(2)]")
print("=" * 70)


def extract_features(image_path: str) -> np.ndarray:
    """
    Extract 9-dim fusion vector for one image.
    Layout:  [m1_sev(5) | m2_jsn(2) | m3_morph(2)]

    m1_sev   : 5-class severity softmax from Model 1 (CBAM TorchScript)
    m2_jsn   : 2-bin JSN representation from Model 2 (Keras) — see jsn_to_2bin()
    m3_morph : Osteophytes + Sclerosis sigmoid from Model 3 (MorphAttention)
               dimensions [0] and [1] of the 4-class output
    """
    img_pt = preprocess_for_pt(image_path)

    # M1 — severity
    with torch.no_grad():
        sev_probs = F.softmax(model1(img_pt), dim=1).cpu().numpy()[0]  # (5,)

    # M2 — JSN 2-bin
    img_tf   = preprocess_for_tf(image_path)
    raw_m2   = model2_tf.predict(img_tf, verbose=0)
    jsn_bins = jsn_to_2bin(raw_m2)                                      # (2,)

    # M3 — morphology (use only Osteophytes [0] + Sclerosis [1])
    with torch.no_grad():
        all_morph = torch.sigmoid(model3(img_pt)).cpu().numpy()[0]      # (4,)
    morph2 = all_morph[:2].astype(np.float32)                           # (2,)

    return np.concatenate([sev_probs, jsn_bins, morph2])                # (9,)


print("[STEP 6/10] extract_features() defined — outputs 9-dim Vfusion.")

# ==============================================================================
# STEP 7 — Build Feature Dataset + Stratified 500-Sample Tally
# ==============================================================================
print("\n" + "=" * 70)
print("[STEP 7/10] Building feature dataset ...")
print("=" * 70)

all_paths  = [p for p, _ in pool_manifest]
raw_labels = [g for _, g in pool_manifest]

CLASS_IDS = sorted(set(raw_labels))
grade2idx = {g: i for i, g in enumerate(CLASS_IDS)}
all_labels = [grade2idx[g] for g in raw_labels]
NUM_CLS    = len(CLASS_IDS)

print(f"  Pool: {len(all_paths)} images | Classes: {CLASS_IDS}")

# ── Stratified 500-image tally ─────────────────────────────────────────────────
tally_paths, tally_labels = [], []
per_cls = max(1, TALLY_N // NUM_CLS)
for cid in CLASS_IDS:
    pool_idx = [i for i, l in enumerate(all_labels) if l == grade2idx[cid]]
    chosen   = random.sample(pool_idx, min(per_cls, len(pool_idx)))
    tally_paths  += [all_paths[i]  for i in chosen]
    tally_labels += [all_labels[i] for i in chosen]
actual_tally = len(tally_paths)
print(f"  Tally: {actual_tally} images | dist: {dict(sorted(Counter(tally_labels).items()))}")

# ── Feature extraction for ALL images ─────────────────────────────────────────
print(f"\n  Extracting Vfusion for ALL {len(all_paths)} images (9D each) ...")
all_features: list = []
for i, path in enumerate(all_paths, 1):
    try:
        feats = extract_features(path)
    except Exception as ex:
        print(f"    WARN [{i}] {path}: {ex} — zeros used.")
        feats = np.zeros(FUSION_DIM, dtype=np.float32)
    all_features.append(feats)
    if i % 200 == 0 or i == len(all_paths):
        print(f"    {i}/{len(all_paths)} ...")

all_features = np.array(all_features, dtype=np.float32)
print(f"  Feature matrix: {all_features.shape}")

# ── Tally feature matrix ───────────────────────────────────────────────────────
path2idx  = {p: i for i, p in enumerate(all_paths)}
tally_idx = [path2idx[p] for p in tally_paths]
X_tally   = all_features[tally_idx]
y_tally   = np.array(tally_labels)

# ── Train / Val / Test split ───────────────────────────────────────────────────
X_tv, X_test, y_tv, y_test = train_test_split(
    all_features, all_labels, test_size=0.15, stratify=all_labels, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.176, stratify=y_tv, random_state=42)

y_train_np = np.array(y_train)
y_val_np   = np.array(y_val)
y_test_np  = np.array(y_test)

print(f"\n  Split → Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"  Train dist: {dict(sorted(Counter(y_train).items()))}")

# ==============================================================================
# STEP 8 — Intermediate Fusion MLP  (Vfusion → Pmeta)
# ==============================================================================
print("\n" + "=" * 70)
print("[STEP 8/10] Intermediate Fusion MLP  (Vfusion 9D → Pmeta 5D) ...")
print("=" * 70)

# ── Dataset & Loaders ─────────────────────────────────────────────────────────
class FusionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


train_ds = FusionDataset(X_train, y_train)
val_ds   = FusionDataset(X_val,   y_val)
test_ds  = FusionDataset(X_test,  y_test)
tally_ds = FusionDataset(X_tally, y_tally)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)
tally_loader = DataLoader(tally_ds, batch_size=BATCH_SIZE)

# Class weights (handle imbalance)
cnt_s      = Counter(y_train)
cls_wts    = [len(y_train) / (NUM_CLS * cnt_s.get(i, 1)) for i in range(NUM_CLS)]
cls_tensor = torch.FloatTensor(cls_wts).to(DEVICE)
print(f"  Class weights: {[round(w, 3) for w in cls_wts]}")


# ── Lean MLP: 9 → 64 → 32 → 5 ────────────────────────────────────────────────
class IntermediateFusionMLP(nn.Module):
    """
    Lightweight intermediate fusion MLP.
    Maps Vfusion ∈ ℝ⁹ → abstract meta-probability Pmeta ∈ ℝ⁵.

    Architecture:  9 → BN → 64 → ReLU → Dropout(0.4) →
                          32 → ReLU → Dropout(0.3) → 5 (logits)
    Dropout is intentionally high to prevent overfitting on the small
    9-dim input; label smoothing (0.05) regularises the CE loss.
    """
    def __init__(self, in_dim: int = FUSION_DIM, hidden: int = MLP_HIDDEN,
                 out_dim: int = MLP_OUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(in_dim),
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden // 2, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)        # raw logits — apply softmax externally


mlp = IntermediateFusionMLP().to(DEVICE)
print(f"\n  IntermediateFusionMLP:\n{mlp}")

# ── Training ──────────────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(weight=cls_tensor, label_smoothing=0.05)
optimiser = optim.AdamW(mlp.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=NUM_EPOCHS, eta_min=1e-5)

best_wts      = copy.deepcopy(mlp.state_dict())
best_val_loss = float("inf")
no_improve    = 0
history       = {"tr_loss": [], "va_loss": [], "tr_acc": [], "va_acc": []}

print(f"\n  Training for up to {NUM_EPOCHS} epochs (patience={PATIENCE}) ...")
for epoch in range(NUM_EPOCHS):
    # Train
    mlp.train()
    tr_loss = tr_correct = tr_total = 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimiser.zero_grad()
        loss = criterion(mlp(X_b), y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(mlp.parameters(), 1.0)
        optimiser.step()
        tr_loss    += loss.item()
        tr_correct += (mlp(X_b).argmax(1) == y_b).sum().item()
        tr_total   += y_b.size(0)
    scheduler.step()

    # Validate
    mlp.eval()
    va_loss = va_correct = va_total = 0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            out = mlp(X_b)
            va_loss    += criterion(out, y_b).item()
            va_correct += (out.argmax(1) == y_b).sum().item()
            va_total   += y_b.size(0)

    t_l = tr_loss / len(train_loader);  t_a = tr_correct / tr_total
    v_l = va_loss / len(val_loader);    v_a = va_correct / va_total
    history["tr_loss"].append(t_l);  history["va_loss"].append(v_l)
    history["tr_acc"].append(t_a);   history["va_acc"].append(v_a)

    print(f"  E{epoch+1:03d}/{NUM_EPOCHS} | "
          f"Tr: loss={t_l:.4f} acc={t_a:.4f} | "
          f"Va: loss={v_l:.4f} acc={v_a:.4f} | no_imp={no_improve}/{PATIENCE}")

    if v_l < best_val_loss:
        best_val_loss = v_l
        best_wts      = copy.deepcopy(mlp.state_dict())
        no_improve    = 0
        torch.save(mlp.state_dict(), os.path.join(SAVE_DIR, "best_mlp_intermediate.pth"))
    else:
        no_improve += 1
    if no_improve >= PATIENCE:
        print(f"  Early stop at epoch {epoch+1}.")
        break

mlp.load_state_dict(best_wts)
torch.save(mlp.state_dict(), os.path.join(SAVE_DIR, "final_mlp_intermediate.pth"))
print(f"  MLP best val loss: {best_val_loss:.4f} | saved ✓")

# ── Generate Pmeta for all splits ─────────────────────────────────────────────
def get_pmeta(X_np: np.ndarray) -> np.ndarray:
    """Run MLP inference, return Pmeta softmax probabilities (N, 5)."""
    mlp.eval()
    out_list = []
    loader   = DataLoader(
        FusionDataset(X_np, np.zeros(len(X_np), dtype=int)),
        batch_size=BATCH_SIZE
    )
    with torch.no_grad():
        for X_b, _ in loader:
            logits = mlp(X_b.to(DEVICE))
            out_list.append(F.softmax(logits, dim=1).cpu().numpy())
    return np.vstack(out_list)


Pmeta_train = get_pmeta(X_train)
Pmeta_val   = get_pmeta(X_val)
Pmeta_test  = get_pmeta(X_test)
Pmeta_tally = get_pmeta(X_tally)

print(f"  Pmeta shapes → train: {Pmeta_train.shape} | val: {Pmeta_val.shape}")

# ==============================================================================
# STEP 9 — Late Fusion + XGBoost Arbiter  (Xboost = Vfusion ⊕ Pmeta → XGB)
# ==============================================================================
print("\n" + "=" * 70)
print("[STEP 9/10] Late Fusion Decision Vector + XGBoost Arbiter ...")
print("=" * 70)

# ── Late fusion: Xboost = [Vfusion(9) ⊕ Pmeta(5)] ∈ ℝ¹⁴ ──────────────────────
scaler = StandardScaler()

def build_xboost_vec(Vfusion: np.ndarray, Pmeta: np.ndarray) -> np.ndarray:
    """Concatenate fusion vector with meta-probabilities."""
    return np.hstack([Vfusion, Pmeta])


Xb_train_raw = build_xboost_vec(X_train, Pmeta_train)
Xb_val_raw   = build_xboost_vec(X_val,   Pmeta_val)
Xb_test_raw  = build_xboost_vec(X_test,  Pmeta_test)
Xb_tally_raw = build_xboost_vec(X_tally, Pmeta_tally)

Xb_train = scaler.fit_transform(Xb_train_raw)
Xb_val   = scaler.transform(Xb_val_raw)
Xb_test  = scaler.transform(Xb_test_raw)
Xb_tally = scaler.transform(Xb_tally_raw)

print(f"  Xboost shape: {Xb_train.shape}  [Vfusion(9) ⊕ Pmeta(5) = 14D, scaled]")

# ── XGBoost training ──────────────────────────────────────────────────────────
#   Tuned for efficiency: hist tree method, moderate depth, early stopping
xgb = XGBClassifier(
    n_estimators         = 500,
    max_depth            = 5,
    learning_rate        = 0.05,
    subsample            = 0.8,
    colsample_bytree     = 0.8,
    min_child_weight     = 3,
    gamma                = 0.1,
    reg_alpha            = 0.05,
    reg_lambda           = 1.0,
    eval_metric          = "mlogloss",
    tree_method          = "hist",
    device               = "cuda" if torch.cuda.is_available() else "cpu",
    early_stopping_rounds= 30,
    random_state         = 42,
    verbosity            = 1,
)

xgb.fit(
    Xb_train, y_train_np,
    eval_set=[(Xb_val, y_val_np)],
    verbose=50,
)

val_acc  = accuracy_score(y_val_np,  xgb.predict(Xb_val))
test_acc = accuracy_score(y_test_np, xgb.predict(Xb_test))
print(f"\n  XGBoost Val Acc  : {val_acc*100:.2f}%")
print(f"  XGBoost Test Acc : {test_acc*100:.2f}%")

# Save XGB + scaler
with open(os.path.join(SAVE_DIR, "xgboost_arbiter.pkl"), "wb") as f:
    pickle.dump({"xgb": xgb, "scaler": scaler}, f)
print("  Saved xgboost_arbiter.pkl ✓")

# ==============================================================================
# STEP 10 — Evaluation, Grad-CAM, and Dashboard
# ==============================================================================
print("\n" + "=" * 70)
print("[STEP 10/10] Evaluation + Grad-CAM + Dashboard ...")
print("=" * 70)

# ── 10.1  Training Curves ─────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["tr_loss"], label="Train", color="steelblue")
ax1.plot(history["va_loss"], label="Val",   color="orange")
ax1.set_title("Intermediate MLP — Loss"); ax1.legend(); ax1.set_xlabel("Epoch")

ax2.plot(history["tr_acc"],  label="Train", color="seagreen")
ax2.plot(history["va_acc"],  label="Val",   color="crimson")
ax2.set_title("Intermediate MLP — Accuracy"); ax2.legend(); ax2.set_xlabel("Epoch")

plt.suptitle("Intermediate Fusion MLP Training", fontweight="bold")
plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "mlp_training_curves.png"), dpi=150)
plt.show()
print("  Saved mlp_training_curves.png ✓")

# ── 10.2  XGBoost Feature Importance ──────────────────────────────────────────
feat_names = (
    [f"sev_G{i}" for i in range(5)] +
    ["jsn_narrow", "jsn_normal"] +
    ["morph_osteo", "morph_scler"] +
    [f"pmeta_G{i}" for i in range(5)]
)
importances = xgb.feature_importances_
idx_sorted  = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(importances)),
       importances[idx_sorted], color="steelblue", alpha=0.85)
ax.set_xticks(range(len(importances)))
ax.set_xticklabels([feat_names[i] for i in idx_sorted], rotation=45, ha="right", fontsize=9)
ax.set_title("XGBoost Feature Importance  (Xboost = Vfusion ⊕ Pmeta)", fontweight="bold")
ax.set_ylabel("Importance Score")
plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "xgb_feature_importance.png"), dpi=150)
plt.show()
print("  Saved xgb_feature_importance.png ✓")

# ── 10.3  Confusion Matrices (Test + Tally) ────────────────────────────────────
y_test_pred  = xgb.predict(Xb_test)
y_tally_pred = xgb.predict(Xb_tally)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
dlabels = [f"G{CLASS_IDS[i]}" for i in range(NUM_CLS)]
for ax, (true, pred, title) in zip(axes, [
    (y_test_np,        y_test_pred,  "Test Set"),
    (np.array(y_tally), y_tally_pred, f"Tally-{actual_tally}"),
]):
    ConfusionMatrixDisplay(confusion_matrix(true, pred), display_labels=dlabels) \
        .plot(cmap=plt.cm.Blues, values_format="d", ax=ax, colorbar=False)
    ax.set_title(f"XGBoost Arbiter — {title}\n"
                 f"Acc: {accuracy_score(true, pred)*100:.2f}%", fontweight="bold")
plt.suptitle("Confusion Matrices — XGBoost Final Arbiter", fontweight="bold")
plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "confusion_matrices_xgb.png"), dpi=150)
plt.show()
print("  Saved confusion_matrices_xgb.png ✓")

# ── 10.4  Classification Report ───────────────────────────────────────────────
tnames = [f"Gr{CLASS_IDS[i]} {SEVERITY_NAMES[i]}" for i in range(NUM_CLS)]
report_test  = classification_report(y_test_np,        y_test_pred,  target_names=tnames)
report_tally = classification_report(np.array(y_tally), y_tally_pred, target_names=tnames)
rpt_path = os.path.join(SAVE_DIR, "classification_report_xgb.txt")
with open(rpt_path, "w") as f:
    f.write("=== XGBoost Arbiter — Test Set ===\n" + report_test + "\n\n")
    f.write(f"=== XGBoost Arbiter — Tally-{actual_tally} ===\n" + report_tally)
print("  Saved classification_report_xgb.txt ✓")
print("\n  [Test Set Report]")
print(report_test)

# ==============================================================================
# GRAD-CAM  — Explainability Engine on Model 1 Backbone
# ==============================================================================
print("\n" + "─" * 70)
print("  GRAD-CAM  — Explainability on Model 1 (EfficientNet-B5 + CBAM)")
print("  Target: predicted KL severity class")
print("─" * 70)

# ── Hook-based Grad-CAM for TorchScript models ────────────────────────────────
#   TorchScript does not support hooks directly on named submodules.
#   Strategy: use the .graph to find the last Conv2d-like op, then fall back
#   to running the raw (non-scripted) backbone if available.
#   For robustness in a TorchScript environment, we register hooks on the
#   underlying module by temporarily de-scripting via torch.jit.export trick
#   or by reloading the model without script.  The cleanest approach here is
#   to implement a lightweight hook-compatible wrapper that re-runs the same
#   forward pass without torch.no_grad() for the target class gradient.

class GradCAMExtractor:
    """
    Grad-CAM extractor for a TorchScript model (model1).

    Because TorchScript models don't expose named modules via model.named_modules(),
    we use a forward-hook approach on the model directly, capturing all intermediate
    Conv2D outputs, and selecting the last spatial one as the target layer.

    If that fails (e.g. fully-scripted with no hooks), we fall back to a
    gradient-based saliency map (GradInput) which still provides spatial
    visualisation aligned with Grad-CAM's intent.
    """

    def __init__(self, model: torch.nn.Module, device: torch.device):
        self.model     = model
        self.device    = device
        self.gradients = None
        self.activations= None
        self._hooks    = []

    def _register_hooks(self):
        """Register forward & backward hooks on the last-found Conv2d."""
        target_layer = None

        # Try to find a named Conv layer in the underlying module tree
        try:
            for name, module in self.model.named_modules():
                if isinstance(module, nn.Conv2d):
                    target_layer = module
        except Exception:
            pass

        if target_layer is None:
            return False   # hooks unavailable

        def save_activation(module, inp, out):
            self.activations = out.detach()

        def save_gradient(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()

        self._hooks.append(target_layer.register_forward_hook(save_activation))
        self._hooks.append(target_layer.register_backward_hook(save_gradient))
        return True

    def _remove_hooks(self):
        for h in self._hooks:
            h.remove()
        self._hooks = []

    def compute(self, img_tensor: torch.Tensor, target_class: int) -> np.ndarray:
        """
        Returns a Grad-CAM heatmap (H,W) normalised to [0,1].
        Falls back to gradient saliency if hooks unavailable.
        """
        self._remove_hooks()
        hooks_ok = self._register_hooks()

        img_tensor = img_tensor.to(self.device)
        img_tensor.requires_grad_(True)

        self.model.eval()
        # Forward pass (keep graph for gradients)
        logits = self.model(img_tensor)
        if isinstance(logits, (tuple, list)):
            logits = logits[0]

        score = logits[0, target_class]
        self.model.zero_grad()
        score.backward()

        if hooks_ok and self.activations is not None and self.gradients is not None:
            # Standard Grad-CAM (Eq. 29-30 in paper)
            weights = self.gradients.mean(dim=(2, 3), keepdim=True)  # α_k^c
            cam     = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
            cam     = cam.squeeze().cpu().numpy()
        else:
            # Fallback: gradient × input saliency
            grad   = img_tensor.grad.data.abs().squeeze()
            cam    = grad.mean(0).cpu().numpy()   # average over channels

        # Normalise to [0,1]
        cam -= cam.min()
        if cam.max() > 0:
            cam /= cam.max()

        self._remove_hooks()
        return cam


def overlay_gradcam(image_path: str, cam: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    """
    Resize cam to image size, apply JET colormap, overlay on original image.
    Returns RGB uint8 (H,W,3) overlay.
    """
    orig = np.array(Image.open(image_path).convert("RGB").resize((224, 224)))

    cam_resized = cv2.resize(cam, (224, 224))
    heatmap     = (cm.jet(cam_resized)[:, :, :3] * 255).astype(np.uint8)
    overlay     = (alpha * heatmap + (1 - alpha) * orig).astype(np.uint8)
    return overlay


gradcam_extractor = GradCAMExtractor(model1, DEVICE)

# ── 10.5  Inference + Grad-CAM Dashboard (5 tally samples) ───────────────────
print(f"\n  Generating Grad-CAM dashboards for 5 random tally samples ...")
sample_indices = random.sample(range(actual_tally), min(5, actual_tally))

for si, idx in enumerate(sample_indices, 1):
    img_path   = tally_paths[idx]
    true_idx   = int(y_tally[idx])
    true_grade = CLASS_IDS[true_idx]

    # ── Pipeline inference for this sample ────────────────────────────────────
    vfusion_np = X_tally[idx]                          # 9D
    pmeta_np   = Pmeta_tally[idx]                      # 5D
    xboost_np  = scaler.transform(
        build_xboost_vec(vfusion_np[None], pmeta_np[None])
    )
    pred_idx   = int(xgb.predict(xboost_np)[0])
    pred_grade = CLASS_IDS[pred_idx]
    xgb_proba  = xgb.predict_proba(xboost_np)[0]      # (5,)
    correct    = pred_grade == true_grade

    # ── Grad-CAM on Model 1 for predicted class ────────────────────────────────
    img_tensor = preprocess_for_pt(img_path)
    cam_map    = gradcam_extractor.compute(img_tensor.clone(), target_class=pred_idx)
    gradcam_overlay = overlay_gradcam(img_path, cam_map)

    # ── Decompose Vfusion for display ─────────────────────────────────────────
    sev_probs  = vfusion_np[:5]
    jsn_vals   = vfusion_np[5:7]
    morph_vals = vfusion_np[7:9]
    sev_names  = SEVERITY_NAMES[:NUM_CLS]

    # ── Plot: 5-panel dashboard ────────────────────────────────────────────────
    fig = plt.figure(figsize=(22, 5))
    gs  = gridspec.GridSpec(1, 5, width_ratios=[1.1, 1.1, 0.9, 0.9, 1.2])

    # Panel 0 — Original X-Ray
    ax0 = plt.subplot(gs[0])
    try:
        ax0.imshow(np.array(Image.open(img_path).convert("RGB").resize((224, 224))))
    except Exception:
        ax0.text(0.5, 0.5, "N/A", ha="center", va="center",
                 transform=ax0.transAxes, fontsize=12)
    ax0.set_title(f"Input X-Ray\nTrue: G{true_grade} {SEVERITY_NAMES[true_idx]}",
                  fontweight="bold", fontsize=9)
    ax0.axis("off")

    # Panel 1 — Grad-CAM Overlay
    ax1 = plt.subplot(gs[1])
    ax1.imshow(gradcam_overlay)
    ax1.set_title(f"Grad-CAM (Model 1)\nTarget: G{pred_grade} "
                  f"{'✓ CORRECT' if correct else '✗ WRONG'}",
                  fontweight="bold", fontsize=9,
                  color="#1a7a2e" if correct else "#c0392b")
    ax1.axis("off")

    # Panel 2 — XGBoost Severity probabilities
    ax2 = plt.subplot(gs[2])
    bar_colors = ['#27ae60' if i == pred_idx else '#2980b9' for i in range(NUM_CLS)]
    bars = ax2.bar(sev_names, xgb_proba, color=bar_colors, alpha=0.85)
    ax2.set_ylim(0, 1.1)
    ax2.axhline(0.5, color="gray", linestyle="--", lw=0.8)
    ax2.set_title(f"XGBoost Severity\nPred: G{pred_grade}", fontweight="bold", fontsize=9)
    ax2.set_ylabel("Probability"); ax2.tick_params(axis="x", labelsize=7, rotation=20)
    for bar in bars:
        h = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
                 f"{h:.2f}", ha="center", va="bottom", fontsize=7)

    # Panel 3 — Fusion Vector Components
    ax3 = plt.subplot(gs[3])
    comp_labels = [f"S{i}" for i in range(5)] + ["JSN↓", "JSN↑"] + ["Osteo", "Scler"]
    comp_vals   = vfusion_np
    comp_colors = (
        ["#2980b9"] * 5 +
        ["#8e44ad", "#8e44ad"] +
        ["#e67e22", "#e67e22"]
    )
    ax3.barh(comp_labels[::-1], comp_vals[::-1], color=comp_colors[::-1], alpha=0.8)
    ax3.set_xlim(0, 1.1)
    ax3.axvline(0.5, color="gray", linestyle="--", lw=0.8)
    ax3.set_title("Vfusion Components\n(9D)", fontweight="bold", fontsize=9)
    ax3.tick_params(axis="y", labelsize=8)
    for i, v in enumerate(comp_vals[::-1]):
        ax3.text(v + 0.02, i, f"{v:.2f}", va="center", fontsize=7)

    # Panel 4 — Text Report
    ax4 = plt.subplot(gs[4])
    ax4.axis("off")

    jsn_label = "NARROW (≥0.5)" if jsn_vals[0] >= 0.5 else "normal (<0.5)"
    morph_osteo = "POSITIVE" if morph_vals[0] > 0.5 else "negative"
    morph_scler = "POSITIVE" if morph_vals[1] > 0.5 else "negative"

    report_str = (
        f"═══ ENSEMBLE REPORT ═══\n\n"
        f"True Grade : G{true_grade}  ({SEVERITY_NAMES[true_idx]})\n"
        f"XGB Pred   : G{pred_grade}  ({SEVERITY_NAMES[pred_idx]})\n"
        f"Result     : {'✓ CORRECT' if correct else '✗ WRONG'}\n"
        f"Confidence : {xgb_proba[pred_idx]*100:.1f}%\n\n"
        f"─── Vfusion (9D) ───\n"
        f"[Severity probs]\n"
        + "".join(
            f"  G{i} {SEVERITY_NAMES[i][:6]:<6}: {sev_probs[i]:.3f}\n"
            for i in range(NUM_CLS)
        )
        + f"\n[JSN 2-bin]\n"
        f"  Narrow    : {jsn_vals[0]:.3f}  {jsn_label}\n"
        f"  Normal    : {jsn_vals[1]:.3f}\n\n"
        f"[Morphology]\n"
        f"  Osteophyte: {morph_vals[0]:.3f}  {morph_osteo}\n"
        f"  Sclerosis : {morph_vals[1]:.3f}  {morph_scler}\n\n"
        f"─── Pmeta (5D MLP) ───\n"
        + "".join(
            f"  G{i}: {pmeta_np[i]:.3f}\n"
            for i in range(NUM_CLS)
        )
        + f"\n─── Grad-CAM ───\n"
        f"  Target class  : G{pred_grade}\n"
        f"  CAM max value : {cam_map.max():.3f}\n"
        f"  CAM mean      : {cam_map.mean():.3f}\n"
    )

    ax4.text(0.03, 0.98, report_str, fontsize=7.5, family="monospace",
             verticalalignment="top", transform=ax4.transAxes,
             bbox=dict(boxstyle="round,pad=0.4", facecolor="#f0f4f8",
                       edgecolor="#8da0bb", alpha=0.95))

    plt.suptitle(
        f"KOA Meta-Ensemble Inference  |  Tally Sample {si}/5  "
        f"[True: G{true_grade}  XGB: G{pred_grade}  "
        f"{'CORRECT ✓' if correct else 'WRONG ✗'}]",
        fontweight="bold", fontsize=11
    )
    plt.tight_layout(rect=[0, 0, 1, 0.94])

    save_path = os.path.join(SAVE_DIR, f"dashboard_gradcam_{si}.png")
    fig.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"  Dashboard {si}/5 saved → {save_path}")

# ── 10.6  Final Summary ────────────────────────────────────────────────────────
print(f"\n{'=' * 70}")
print("[DONE] FINAL RESULTS")
print(f"{'=' * 70}")
print(f"\n  Architecture:")
print(f"    Sub-models     : 3  (M1=severity, M2=JSN→2bin, M3=morphology)")
print(f"    Fusion vector  : {FUSION_DIM}D  (sev5 + jsn2 + morph2)")
print(f"    Intermediate   : MLP  {FUSION_DIM}→{MLP_HIDDEN}→{MLP_HIDDEN//2}→{MLP_OUT}  (Pmeta)")
print(f"    Late fusion    : Xboost = Vfusion({FUSION_DIM}) ⊕ Pmeta({MLP_OUT}) = {XGBOOST_INPUT_DIM}D")
print(f"    Final arbiter  : XGBoost only")
print(f"\n  XGBoost Performance:")
print(f"    Val Acc   : {val_acc*100:.2f}%")
print(f"    Test Acc  : {test_acc*100:.2f}%")
print(f"    Tally Acc : {accuracy_score(np.array(y_tally), y_tally_pred)*100:.2f}%")
print(f"\n  Saved outputs:")
for fname in sorted(os.listdir(SAVE_DIR)):
    mb = os.path.getsize(os.path.join(SAVE_DIR, fname)) / 1e6
    print(f"    {fname:<45} {mb:.2f} MB")
print(f"\n{'=' * 70}")
print("[DONE] MetaKneeEnsemble v3 pipeline complete.")